In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import warnings
import massbalancemachine as mbm
import logging
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import json 

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.utils import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

print(torch.cuda.is_available())   # True
print(torch.cuda.device_count())   # 0  ← wrong$

if torch.cuda.device_count()==0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")


In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"TF_REGION"
CACHE_DIR = BASE_DIR / "LSTM_cache_TL/Transformer_exp"

# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

# make sure BASE_DIR exists
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")
print(f"Cache directory for models: {CACHE_DIR}")

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

### Monthly datasets:
Build monthly datasets for sequence datasets. 

In [ ]:
import geopandas as gpd

rgi_gdf = gpd.read_file(cfg.dataPath / RGI_V6_ROOT /
                        RGI_REGIONS['13']['folder'] /
                        RGI_REGIONS['13']['file'])[['RGIId', 'O2Region']]

glacier_dict = (dfs['13'][['GLACIER', 'RGIId']].drop_duplicates().merge(
    rgi_gdf, on='RGIId', how='left').set_index('GLACIER').to_dict('index'))

dfs['13']['O2Region'] = dfs['13']['GLACIER'].map({
    k: v['O2Region']
    for k, v in glacier_dict.items()
})

dfs['13'].groupby('O2Region').size().rename('n_measurements')

In [ ]:
dfs['13'].groupby('O2Region').agg(n_measurements=('GLACIER', 'size'),
                                  glaciers=('GLACIER',
                                            lambda x: sorted(x.unique())))

In [ ]:
# Build O2Region -> color mapping
o2_regions = sorted(
    set(v['O2Region'] for v in glacier_dict.values()
        if v['O2Region'] is not None))
region_colors = [
    "red", "blue", "green", "purple", "orange", "darkred", "cadetblue",
    "darkgreen", "darkpurple", "pink", "gray", "black"
]
o2_color_map = {
    r: region_colors[i % len(region_colors)]
    for i, r in enumerate(o2_regions)
}

# Map each glacier to its O2Region color
color_map = {
    glacier: o2_color_map[info['O2Region']]
    for glacier, info in glacier_dict.items() if info['O2Region'] is not None
}

m = plot_stakes_folium(dfs['13'], color_map=color_map, tooltip_col='O2Region')
m

In [ ]:
SOURCE_REGIONS = [  # regions for "foundation model"
    'CH',
    'NOR',
    'ISL',
    'FR',
]
TARGET_REGIONS = ['CENTRALASIA']

paths_multi = {
    "EU_US_CANADA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_EU_US_CANADA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_EU_US_CANADA.nc"),
    },
    "HMA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
    },
}

run_flag_by_code = {
    'CH': False,
    'NOR': False,
    'ISL': False,
    'FR': False,
    'CENTRALASIA': False,
}

monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

res_xreg_by_source = {
    region: monthly_cache[region]
    for region in SOURCE_REGIONS + TARGET_REGIONS
}

months_head_pad = res_xreg_by_source["CH"]['months_head_pad']
months_tail_pad = res_xreg_by_source["CH"]['months_tail_pad']
print(f"months_head_pad: {months_head_pad}")
print(f"months_tail_pad: {months_tail_pad}")

In [ ]:
# Add O2Region to CENTRALASIA monthly data
data_ca = monthly_cache['CENTRALASIA']['data_monthly'].copy()
data_ca['O2Region'] = data_ca['GLACIER'].map({
    k: v['O2Region']
    for k, v in glacier_dict.items()
})

# Split
data_CA_12 = data_ca[data_ca['O2Region'].isin(['1', '2'])].copy()
data_CA_3 = data_ca[data_ca['O2Region'] == '3'].copy()
data_CA_4 = data_ca[data_ca['O2Region'] == '4'].copy()

print(f"Number of measurements in O2Region 1+2: {len(data_CA_12)}")
print(f"Number of measurements in O2Region 3: {len(data_CA_3)}")
print(f"Number of measurements in O2Region 4: {len(data_CA_4)}")

### Training glaciers:

#### Automatic picking:

In [ ]:
ca_glaciers_12 = set(data_CA_12['GLACIER'].unique())
ca_glaciers_3 = set(data_CA_3['GLACIER'].unique())
ca_glaciers_4 = set(data_CA_4['GLACIER'].unique())

res_xreg_by_source_ext = {
    **res_xreg_by_source,
    'CA_12': {
        'data_monthly':
        data_CA_12,
        'data_monthly_aug':
        res_xreg_by_source['CENTRALASIA']['data_monthly_aug']
        [res_xreg_by_source['CENTRALASIA']['data_monthly_aug']['GLACIER'].isin(
            ca_glaciers_12)],
        'months_head_pad':
        res_xreg_by_source['CENTRALASIA']['months_head_pad'],
        'months_tail_pad':
        res_xreg_by_source['CENTRALASIA']['months_tail_pad'],
    },
    'CA_3': {
        'data_monthly':
        data_CA_3,
        'data_monthly_aug':
        res_xreg_by_source['CENTRALASIA']['data_monthly_aug']
        [res_xreg_by_source['CENTRALASIA']['data_monthly_aug']['GLACIER'].isin(
            ca_glaciers_3)],
        'months_head_pad':
        res_xreg_by_source['CENTRALASIA']['months_head_pad'],
        'months_tail_pad':
        res_xreg_by_source['CENTRALASIA']['months_tail_pad'],
    },
    'CA_4': {
        'data_monthly':
        data_CA_4,
        'data_monthly_aug':
        res_xreg_by_source['CENTRALASIA']['data_monthly_aug']
        [res_xreg_by_source['CENTRALASIA']['data_monthly_aug']['GLACIER'].isin(
            ca_glaciers_4)],
        'months_head_pad':
        res_xreg_by_source['CENTRALASIA']['months_head_pad'],
        'months_tail_pad':
        res_xreg_by_source['CENTRALASIA']['months_tail_pad'],
    },
}

In [ ]:
# remap cache keys to what the function expects
res_xreg_remapped = {
    region: {
        "df_train": res_xreg_by_source[region]['data_monthly'],
        "df_test": res_xreg_by_source[region]['data_monthly_aug'],
    }
    for region in SOURCE_REGIONS
}

scaler_m, scaler_s, scaler_all = build_global_scalers_multi_source_simple(
    res_xreg_remapped,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
)

blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
    res_xreg_remapped,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    scaler_m=scaler_m,
    scaler_s=scaler_s,
    seed=cfg.seed,
)
print(f"blur_m={blur_m:.4f}  blur_s={blur_s:.4f}  blur_joint={blur_joint:.4f}")

#### Split glaciers of training sets:

In [ ]:
RUN_GLACIER_RANK = True

# --- whole CENTRALASIA ---
for mode, topo_extra_cols, save_name in [
    ("joint", [], "glacier_ranking_ca_joint.csv"),
]:
    save_path = BASE_DIR / save_name

    if RUN_GLACIER_RANK:
        df_ranked = rank_glaciers_by_distance_to_target(
            res_xreg_by_source=res_xreg_by_source,
            training_regions=SOURCE_REGIONS,
            test_region='CENTRALASIA',
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            scaler_m=scaler_m,
            scaler_s=scaler_s,
            blur_joint=blur_joint,
            seed=cfg.seed,
            mode=mode,
            topo_extra_cols=topo_extra_cols,
        )
        df_ranked.to_csv(save_path, index=False)
        print(f"Saved {mode} ranking -> {save_path}")
    else:
        df_ranked = pd.read_csv(save_path)
        print(f"Loaded {mode} ranking from {save_path}")

    df_ranked_ca_joint = df_ranked

# --- per subregion ---
rankings_ca = {}
for test_region, save_name in [
    ("CA_12", "glacier_ranking_ca12_joint.csv"),
    ("CA_3", "glacier_ranking_ca3_joint.csv"),
    ("CA_4", "glacier_ranking_ca4_joint.csv"),
]:
    save_path = BASE_DIR / save_name

    if RUN_GLACIER_RANK:
        df_ranked = rank_glaciers_by_distance_to_target(
            res_xreg_by_source=res_xreg_by_source_ext,
            training_regions=SOURCE_REGIONS,
            test_region=test_region,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            scaler_m=scaler_m,
            scaler_s=scaler_s,
            blur_joint=blur_joint,
            seed=cfg.seed,
            mode="joint",
            topo_extra_cols=[],
        )
        df_ranked.to_csv(save_path, index=False)
        print(f"Saved {test_region} joint ranking -> {save_path}")
    else:
        df_ranked = pd.read_csv(save_path)
        print(f"Loaded {test_region} joint ranking from {save_path}")

    rankings_ca[test_region] = df_ranked

df_ranked_ca12_joint = rankings_ca["CA_12"]
df_ranked_ca3_joint = rankings_ca["CA_3"]
df_ranked_ca4_joint = rankings_ca["CA_4"]

# --- comparison across subregions ---
for r1, r2 in [("CA_12", "CA_3"), ("CA_12", "CA_4"), ("CA_3", "CA_4")]:
    top10_r1 = set(rankings_ca[r1].head(10)['glacier'])
    top10_r2 = set(rankings_ca[r2].head(10)['glacier'])
    print(
        f"\nTop-10 overlap ({r1} ∩ {r2}): {len(top10_r1 & top10_r2)} glaciers")
    print(f"  {r1} only: {sorted(top10_r1 - top10_r2)}")
    print(f"  {r2} only: {sorted(top10_r2 - top10_r1)}")

## Train transformers:

In [ ]:
gs_logs_dir = BASE_DIR / "logs/Transformer_GS"
logs_gs_transformer = pd.read_csv(
    os.path.join(gs_logs_dir, "transformer_gs_pooled_2026-05-20.csv"))

# best by validation loss
best_row = logs_gs_transformer.sort_values("valid_loss").iloc[0]
print("Best config:")
print(best_row[[
    "d_model", "nhead", "num_layers", "dim_feedforward", "dropout",
    "static_layers", "static_hidden", "static_dropout", "head_dropout", "lr",
    "weight_decay", "valid_loss", "val_rmse_a", "val_rmse_w"
]])

best_params_gs = {
    "Fm":
    int(best_row["Fm"]),
    "Fs":
    int(best_row["Fs"]),
    "d_model":
    int(best_row["d_model"]),
    "nhead":
    int(best_row["nhead"]),
    "num_layers":
    int(best_row["num_layers"]),
    "dim_feedforward":
    int(best_row["dim_feedforward"]),
    "dropout":
    float(best_row["dropout"]),
    "static_layers":
    int(best_row["static_layers"]),
    "static_hidden": (None if pd.isna(best_row["static_hidden"]) else int(
        best_row["static_hidden"])),
    "static_dropout": (None if pd.isna(best_row["static_dropout"]) else float(
        best_row["static_dropout"])),
    "head_dropout":
    float(best_row["head_dropout"]),
    "lr":
    float(best_row["lr"]),
    "weight_decay":
    float(best_row["weight_decay"]),
    "two_heads":
    False,
    "loss_spec":
    None,
    "T_max":
    32,
}

print("\nbest_params_gs:")
for k, v in best_params_gs.items():
    print(f"  {k}: {v}")

lstm_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None
}

In [ ]:
FRACS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.4]
N_RANDOM_SEEDS = 10

# --- 1. Build glacier subsets ---
gl_subsets_ca = {}

for test_region, df_ranked in [
    ("CENTRALASIA", df_ranked_ca_joint),
    ("CA_12",       df_ranked_ca12_joint),
    ("CA_3",        df_ranked_ca3_joint),
    ("CA_4",        df_ranked_ca4_joint),
]:
    gl_subsets_ca[test_region] = build_glacier_subsets(
        df_ranked=df_ranked,
        fracs=FRACS,
        n_random_seeds=N_RANDOM_SEEDS,
        seed=cfg.seed,
    )
    print(f"Subsets built for {test_region}")

# convenience aliases
gl_subsets_ca_joint    = gl_subsets_ca["CENTRALASIA"]
gl_subsets_ca12_joint  = gl_subsets_ca["CA_12"]
gl_subsets_ca3_joint   = gl_subsets_ca["CA_3"]
gl_subsets_ca4_joint   = gl_subsets_ca["CA_4"]

# --- 2. Dummy splits ---
dummy_splits = {
    region: {"holdout_glaciers": [], "pool_glaciers": []}
    for region in SOURCE_REGIONS
}

# --- 3. Test datasets ---
df_ca_full = res_xreg_by_source['CENTRALASIA']['data_monthly']
df_ca_aug  = res_xreg_by_source['CENTRALASIA']['data_monthly_aug']

ds_ca_test = build_combined_LSTM_dataset(
    df_loss=df_ca_full,
    df_full=df_ca_aug,
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    months_head_pad=res_xreg_by_source['CENTRALASIA']['months_head_pad'],
    months_tail_pad=res_xreg_by_source['CENTRALASIA']['months_tail_pad'],
    normalize_target=True,
    expect_target=True,
    show_progress=False,
)
print(f"CENTRALASIA test : {len(ds_ca_test)} sequences")

for subregion, glaciers in [
    ("CA_12", ca_glaciers_12),
    ("CA_3",  ca_glaciers_3),
    ("CA_4",  ca_glaciers_4),
]:
    ds = build_combined_LSTM_dataset(
        df_loss=df_ca_full[df_ca_full['GLACIER'].isin(glaciers)],
        df_full=df_ca_aug[ df_ca_aug['GLACIER'].isin(glaciers)],
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        months_head_pad=res_xreg_by_source['CENTRALASIA']['months_head_pad'],
        months_tail_pad=res_xreg_by_source['CENTRALASIA']['months_tail_pad'],
        normalize_target=True,
        expect_target=True,
        show_progress=False,
    )
    print(f"{subregion} test : {len(ds)} sequences")
    # store in named variable
    if subregion == "CA_12":
        ds_ca12_test = ds
    elif subregion == "CA_3":
        ds_ca3_test = ds
    else:
        ds_ca4_test = ds

In [ ]:
# --- 3. Train transformer for each subset ---
TRAIN_FLAG = False
MODEL_DATE = "2026-05-21"

CACHE_DIR_RANK_CA = CACHE_DIR / "ranking_ca_subregions"
os.makedirs(CACHE_DIR_RANK_CA, exist_ok=True)

ranking_results_ca_sub = []

for ranking_label, df_ranked, gl_subsets_cur, ds_test_match, test_label_match in [
    ("ranked_by_ca_joint",  df_ranked_ca_joint,  gl_subsets_ca_joint,  ds_ca_test,  "CENTRALASIA"),
    # ("ranked_by_ca12_joint", df_ranked_ca12_joint, gl_subsets_ca12_joint, ds_ca12_test, "CA_12"),
    # ("ranked_by_ca3_joint",  df_ranked_ca3_joint,  gl_subsets_ca3_joint,  ds_ca3_test,  "CA_3"),
    # ("ranked_by_ca4_joint",  df_ranked_ca4_joint,  gl_subsets_ca4_joint,  ds_ca4_test,  "CA_4"),
]:
    print(f"\n{'='*60}")
    print(f"  Ranking: {ranking_label}  →  eval on {test_label_match}")

    mode       = "joint"
    models_dir = BASE_DIR / "models/ranking_ca_subregions" / test_label_match
    os.makedirs(models_dir, exist_ok=True)

    for pct, subset in gl_subsets_cur.items():
        print(f"\n  Fraction: {pct}%")

        strategies = {
            "closest": subset['closest'],
            **{f"random_{s['seed_idx']}": s['glaciers']
               for s in subset['random']},
        }

        for strategy_name, glaciers in strategies.items():
            print(f"\n  --- {strategy_name} ({len(glaciers)} glaciers) ---")

            assets = build_assets_from_glacier_list(
                glaciers=glaciers,
                df_ranked=df_ranked,
                res_xreg_by_source=res_xreg_by_source_ext,
                monthly_cols=MONTHLY_COLS,
                static_cols=STATIC_COLS,
                cfg=cfg,
                cache_path=str(
                    CACHE_DIR_RANK_CA /
                    f"assets_{test_label_match}_{mode}_{pct}pct_{strategy_name}.joblib"
                ),
                force_recompute=False,
                months_head_pad=months_head_pad,
                months_tail_pad=months_tail_pad,
                src_regions=SOURCE_REGIONS,
            )

            model, model_path, info = train_or_load_one_source_model(
                cfg=cfg,
                key=f"{test_label_match}_{pct}pct_{strategy_name}",
                lstm_assets=assets,
                best_params=best_params_gs,
                device=device,
                models_dir=models_dir,
                prefix=f"transformer_rank_ca",
                train_flag=TRAIN_FLAG,
                force_retrain=False,
                epochs=150,
                date=MODEL_DATE,
                model_type="transformer",
                verbose=False,
            )

            ds_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                assets["ds_train"])
            ds_scaler.make_loaders(
                train_idx=assets["train_idx"],
                val_idx=assets["val_idx"],
                fit_and_transform=True,
                seed=cfg.seed,
                verbose=False,
            )
            ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
                ds_test_match)
            test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
                ds_copy, ds_scaler, batch_size=128, seed=cfg.seed,
            )
            metrics, _ = model.evaluate_with_preds(device, test_dl, ds_copy)

            is_random = strategy_name.startswith("random")
            print(f"    [transformer] [{test_label_match}] "
                  f"RMSE_a={metrics['RMSE_annual']:.3f}  "
                  f"RMSE_w={metrics['RMSE_winter']:.3f}  "
                  f"R2_a={metrics['R2_annual']:.3f}")

            ranking_results_ca_sub.append({
                "ranking_target": ranking_label,
                "model":          "transformer",
                "pct":            pct,
                "strategy":       "random" if is_random else strategy_name,
                "seed_idx":       int(strategy_name.split("_")[1]) if is_random else None,
                "test_region":    test_label_match,
                "n_glaciers":     len(glaciers),
                "n_sequences":    len(assets["ds_train"]),
                **{k: round(v, 3) for k, v in metrics.items()},
            })

    # --- full baseline ---
    print(f"\n  Fraction: 100%")
    full_assets = build_assets_from_glacier_list(
        glaciers=df_ranked['glacier'].tolist(),
        df_ranked=df_ranked,
        res_xreg_by_source=res_xreg_by_source_ext,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        cache_path=str(
            CACHE_DIR_RANK_CA /
            f"assets_{test_label_match}_{mode}_100pct_full.joblib"
        ),
        force_recompute=False,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        src_regions=SOURCE_REGIONS,
    )

    model, _, _ = train_or_load_one_source_model(
        cfg=cfg,
        key=f"{test_label_match}_100pct_full",
        lstm_assets=full_assets,
        best_params=best_params_gs,
        device=device,
        models_dir=models_dir,
        prefix=f"transformer_rank_ca",
        train_flag=TRAIN_FLAG,
        force_retrain=False,
        epochs=150,
        date=MODEL_DATE,
        model_type="transformer",
        verbose=False,
    )

    ds_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        full_assets["ds_train"])
    ds_scaler.make_loaders(
        train_idx=full_assets["train_idx"],
        val_idx=full_assets["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )
    ds_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_test_match)
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_copy, ds_scaler, batch_size=128, seed=cfg.seed,
    )
    metrics, _ = model.evaluate_with_preds(device, test_dl, ds_copy)

    print(f"  transformer_full [{test_label_match}]: "
          f"RMSE_a={metrics['RMSE_annual']:.3f}  "
          f"RMSE_w={metrics['RMSE_winter']:.3f}")

    ranking_results_ca_sub.append({
        "ranking_target": ranking_label,
        "model":          "transformer",
        "pct":            100,
        "strategy":       "transformer_full",
        "seed_idx":       None,
        "test_region":    test_label_match,
        "n_glaciers":     len(df_ranked),
        "n_sequences":    len(full_assets["ds_train"]),
        **{k: round(v, 3) for k, v in metrics.items()},
    })

# --- save ---
df_ranking_results_ca_sub = pd.DataFrame(ranking_results_ca_sub)
df_ranking_results_ca_sub.to_csv(BASE_DIR / "ranking_ca_subregions_results.csv", index=False)
print(df_ranking_results_ca_sub.groupby(
    ["ranking_target", "model", "test_region", "pct", "strategy"]
).mean(numeric_only=True).round(3))

In [ ]:
for model_label in ["transformer"]:
    for ranking_label, test_label_match in [
        ("ranked_by_ca_joint",  "CENTRALASIA"),
        # ("ranked_by_ca12_joint", "CA_12"),
        # ("ranked_by_ca3_joint",  "CA_3"),
        # ("ranked_by_ca4_joint",  "CA_4"),
    ]:
        fig = plot_ranking_results_extended(
            df_results=df_ranking_results_ca_sub,
            ranking_label=ranking_label,
            test_region=test_label_match,
            source_regions=SOURCE_REGIONS,
            n_rand_seeds=N_RANDOM_SEEDS,
            model_label=model_label,
            save_path=BASE_DIR / f"ranking_ca_sub_{ranking_label}_{model_label}",
        )

In [ ]:
FRACS_TO_PLOT = [10, 15, 20, 30]

for mode, df_ranked, gl_subsets_cur in [
    ("joint", df_ranked_ca_joint, gl_subsets_ca_joint),
]:
    models_dir = BASE_DIR / "models/ranking_ca" / mode
    ranking_plot_configs = []

    for pct in FRACS_TO_PLOT:

        # --- closest ---
        assets_close = build_assets_from_glacier_list(
            glaciers=gl_subsets_cur[pct]['closest'],
            df_ranked=df_ranked,
            res_xreg_by_source=res_xreg_by_source,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            cfg=cfg,
            cache_path=str(CACHE_DIR_RANK_CA /
                           f"assets_{mode}_{pct}pct_closest.joblib"),
            months_head_pad=months_head_pad,
            months_tail_pad=months_tail_pad,
            src_regions=SOURCE_REGIONS,
        )
        model_close, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key=f"{pct}pct_closest",
            lstm_assets=assets_close,
            best_params=best_params_gs,
            device=device,
            models_dir=models_dir,
            prefix="transformer_rank",
            train_flag=False,
            date=MODEL_DATE,
            model_type="transformer",
            verbose=False,
        )
        ranking_plot_configs.append(
            (f"Closest {pct}% ({mode})", model_close, assets_close))

        # --- random seed 0 ---
        assets_rnd = build_assets_from_glacier_list(
            glaciers=gl_subsets_cur[pct]['random'][0]['glaciers'],
            df_ranked=df_ranked,
            res_xreg_by_source=res_xreg_by_source,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            cfg=cfg,
            cache_path=str(CACHE_DIR_RANK_CA /
                           f"assets_{mode}_{pct}pct_random_0.joblib"),
            months_head_pad=months_head_pad,
            months_tail_pad=months_tail_pad,
            src_regions=SOURCE_REGIONS,
        )
        model_rnd, _, _ = train_or_load_one_source_model(
            cfg=cfg,
            key=f"{pct}pct_random_0",
            lstm_assets=assets_rnd,
            best_params=best_params_gs,
            device=device,
            models_dir=models_dir,
            prefix="transformer_rank",
            train_flag=False,
            date=MODEL_DATE,
            model_type="transformer",
            verbose=False,
        )
        ranking_plot_configs.append(
            (f"Random {pct}% seed 0 ({mode})", model_rnd, assets_rnd))

    plot_pred_vs_truth_grid(
        plot_configs=ranking_plot_configs,
        ds_test=ds_ca_test,
        device=device,
        cfg=cfg,
        ncols=4,
        ax_xlim=(-8, 6),
        ax_ylim=(-8, 6),
        title=f"Closest vs random ({mode}) → CENTRALASIA",
        save_path=f"figures/ranking_ca_{mode}_pred_vs_truth",
        figsize=(25, 12),
    )